# NeuroGolf submission builder

`solution/` 配下の複数 solution folder を候補集合として読み込み、各 `taskXXX.onnx` の推定 score が最も良い候補から `submission.zip` を作成する notebook。

基本フロー:
1. `solution/<solution_name>/task001.onnx` ... `task400.onnx` を候補として列挙する。
2. 候補全体は full test せず、score 計算だけ行う。
3. task ごとに score が高い順で候補を並べ、暫定 final output を作る。
4. `VERIFY_MODE = "skip"` なら test せずそのまま zip 作成。`"full"` なら最終 output だけ test して失敗 task を next candidate に置き換える。
5. `outputs/submission.zip` を作成する。


In [17]:
from pathlib import Path
import copy
import importlib.util
import json
import math
import os
import re
import shutil
import time
import traceback
import zipfile

import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "neurogolf-2026").is_dir() and (path / "solution").is_dir():
            return path
    raise FileNotFoundError("project root not found")


ROOT = find_project_root()
DATA_DIR = ROOT / "neurogolf-2026"
SOLUTION_ROOT = ROOT / "solution"
OUTPUT_DIR = ROOT / "outputs"
FINAL_OUTPUT_DIR = OUTPUT_DIR / "selected_submission"
PROFILE_DIR = ROOT / "profiles" / "submission-builder"
REPORT_DIR = ROOT / "artifacts" / "submission-builder"
FINAL_ZIP = OUTPUT_DIR / "submission.zip"
MPL_CACHE_DIR = ROOT / ".matplotlib-cache"

FILESIZE_LIMIT = int(1.44 * 1024 * 1024)
TASK_COUNT = 400
TASKS = list(range(1, TASK_COUNT + 1))
VERIFY_MODE = "skip"  # "skip" creates zip from best score only; "full" tests and repairs candidates.
SCORE_CACHE_CSV = REPORT_DIR / "candidate_scores.csv"
SELECTION_CSV = REPORT_DIR / "selected_candidates.csv"
REPAIR_LOG_CSV = REPORT_DIR / "repair_log.csv"

for path in [OUTPUT_DIR, FINAL_OUTPUT_DIR, PROFILE_DIR, REPORT_DIR, MPL_CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

spec = importlib.util.spec_from_file_location(
    "neurogolf_utils",
    DATA_DIR / "neurogolf_utils" / "neurogolf_utils.py",
)
ng = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ng)
ng._NEUROGOLF_DIR = str(DATA_DIR) + "/"

print("ROOT", ROOT)
print("SOLUTION_ROOT", SOLUTION_ROOT)
print("FINAL_OUTPUT_DIR", FINAL_OUTPUT_DIR)
print("FINAL_ZIP", FINAL_ZIP)

ROOT /Users/kaiikeda/Programming/Kaggle/Neuro Golf
SOLUTION_ROOT /Users/kaiikeda/Programming/Kaggle/Neuro Golf/solution
FINAL_OUTPUT_DIR /Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/selected_submission
FINAL_ZIP /Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/submission.zip


## Helpers

Score 計算では full test をしない。`score_network()` が profile JSON を必要とするため、最初に変換できる benchmark input を 1 件だけ流して profile を作る。

In [18]:
def task_num_from_path(path: Path) -> int:
    m = re.fullmatch(r"task(\d{3})\.onnx", path.name)
    if not m:
        raise ValueError(f"unexpected ONNX filename: {path}")
    return int(m.group(1))


def rel(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)


def load_examples(task_num: int):
    with open(DATA_DIR / f"task{task_num:03d}.json") as f:
        return json.load(f)


def iter_benchmarks(examples, include_arc_gen=True):
    keys = ["train", "test"]
    if include_arc_gen:
        keys.append("arc-gen")
    for split in keys:
        for idx, example in enumerate(examples[split]):
            benchmark = ng.convert_to_numpy(example)
            if benchmark is not None:
                yield split, idx, example, benchmark


def first_benchmark(task_num: int):
    for _, _, _, benchmark in iter_benchmarks(load_examples(task_num), include_arc_gen=True):
        return benchmark
    raise ValueError(f"no runnable benchmark for task {task_num:03d}")


def estimated_score(memory, params):
    if memory is None or params is None or memory < 0 or params < 0:
        return None
    return max(1.0, 25.0 - math.log(max(1.0, memory + params)))


def make_profile_session(sanitized_model, profile_prefix: Path):
    options = ort.SessionOptions()
    options.enable_profiling = True
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    options.profile_file_prefix = str(profile_prefix)
    return ort.InferenceSession(sanitized_model.SerializeToString(), options)

## Discover Candidates

`solution/` 直下の各 folder を solution として扱う。400 個揃っていない folder は候補には含めるが、missing count を表示する。

In [19]:
def discover_solution_folders(solution_root=SOLUTION_ROOT):
    folders = [p for p in sorted(solution_root.iterdir()) if p.is_dir()]
    rows = []
    candidates = []
    for folder in folders:
        files = {task_num_from_path(p): p for p in sorted(folder.glob("task*.onnx"))}
        missing = sorted(set(TASKS) - set(files))
        extra = sorted(set(files) - set(TASKS))
        rows.append({
            "solution": folder.name,
            "path": rel(folder),
            "onnx_count": len(files),
            "missing_count": len(missing),
            "extra_count": len(extra),
            "missing_head": missing[:10],
        })
        for task_num, path in files.items():
            if task_num in TASKS:
                candidates.append({
                    "task": task_num,
                    "solution": folder.name,
                    "path": path,
                    "onnx": rel(path),
                    "filesize": path.stat().st_size,
                    "size_ok": path.stat().st_size <= FILESIZE_LIMIT,
                })
    return pd.DataFrame(rows), pd.DataFrame(candidates)


solution_summary, candidates = discover_solution_folders()
print(f"solutions: {len(solution_summary)}")
print(f"candidates: {len(candidates)}")
solution_summary

solutions: 4
candidates: 1600


,solution,path,onnx_count,missing_count,extra_count,missing_head
0,6274.21,solution/6274.21,400,0,0,[]
1,6285.35,solution/6285.35,400,0,0,[]
2,6322.39,solution/6322.39,400,0,0,[]
3,6332.87,solution/6332.87,400,0,0,[]


## Score Only

ここでは候補 ONNX の構造チェック、sanitize、最小 1 run の profile 生成、`score_network()` だけ行う。出力一致の full test はまだしない。

一度計算した score は `artifacts/submission-builder/candidate_scores.csv` に保存する。再計算したい場合は `RECOMPUTE_SCORES = True` にする。

In [20]:
RECOMPUTE_SCORES = False


def score_candidate(row):
    started = time.time()
    path = Path(row["path"])
    task_num = int(row["task"])
    solution = row["solution"]
    result = {
        "task": task_num,
        "solution": solution,
        "onnx": rel(path),
        "filesize": path.stat().st_size,
        "size_ok": path.stat().st_size <= FILESIZE_LIMIT,
        "score_ready": False,
        "memory_bytes": None,
        "params": None,
        "estimated_score": None,
        "profile_path": None,
        "score_error": None,
        "score_seconds": None,
    }
    try:
        if not result["size_ok"]:
            raise ValueError(f"file is over size limit: {result['filesize']} bytes")
        model = onnx.load(path)
        onnx.checker.check_model(model, full_check=True)
        sanitized = ng.sanitize_model(copy.deepcopy(model))
        if sanitized is None:
            raise ValueError("sanitize_model failed")

        profile_prefix = PROFILE_DIR / f"score_{solution}_task{task_num:03d}"
        session = make_profile_session(sanitized, profile_prefix)
        benchmark = first_benchmark(task_num)
        _ = ng.run_network(session, benchmark["input"])
        profile_path = session.end_profiling()
        memory, params = ng.score_network(sanitized, profile_path)
        score = estimated_score(memory, params)
        if score is None:
            raise ValueError(f"score unavailable: memory={memory}, params={params}")

        result.update({
            "score_ready": True,
            "memory_bytes": memory,
            "params": params,
            "estimated_score": score,
            "profile_path": rel(Path(profile_path)),
        })
    except Exception as exc:
        result["score_error"] = repr(exc)
    finally:
        result["score_seconds"] = time.time() - started
    return result


def score_all_candidates(candidates_df, recompute=False):
    if SCORE_CACHE_CSV.exists() and not recompute:
        scores = pd.read_csv(SCORE_CACHE_CSV)
        print(f"loaded score cache: {rel(SCORE_CACHE_CSV)} rows={len(scores)}")
        return scores

    rows = []
    total = len(candidates_df)
    for i, row in candidates_df.sort_values(["task", "solution"]).iterrows():
        print(f"[{len(rows)+1:04d}/{total:04d}] task {int(row['task']):03d} {row['solution']}: scoring")
        scored = score_candidate(row)
        rows.append(scored)
        if scored["score_ready"]:
            print(
                f"  score={scored['estimated_score']:.3f} "
                f"memory={scored['memory_bytes']} params={scored['params']} "
                f"time={scored['score_seconds']:.1f}s"
            )
        else:
            print(f"  score failed: {scored['score_error']}")

    scores = pd.DataFrame(rows)
    scores.to_csv(SCORE_CACHE_CSV, index=False)
    return scores


scores = score_all_candidates(candidates, recompute=RECOMPUTE_SCORES)
scores.head()

loaded score cache: artifacts/submission-builder/candidate_scores.csv rows=1600


,task,solution,onnx,filesize,size_ok,score_ready,memory_bytes,params,estimated_score,profile_path,score_error,score_seconds
0,1,6274.21,solution/6274.21/task001.onnx,1082,True,True,2819.0,24.0,17.047385,profiles/submission-builder/score_6274.21_task...,NaN,0.188303
1,1,6285.35,solution/6285.35/task001.onnx,1082,True,True,2819.0,24.0,17.047385,profiles/submission-builder/score_6285.35_task...,NaN,0.027391
2,1,6322.39,solution/6322.39/task001.onnx,1082,True,True,2819.0,24.0,17.047385,profiles/submission-builder/score_6322.39_task...,NaN,0.020048
3,1,6332.87,solution/6332.87/task001.onnx,1082,True,True,2819.0,24.0,17.047385,profiles/submission-builder/score_6332.87_task...,NaN,0.016240
4,2,6274.21,solution/6274.21/task002.onnx,14560,True,True,97200.0,93.0,13.514518,profiles/submission-builder/score_6274.21_task...,NaN,0.071615


## Rank And Build Initial Output

task ごとに `estimated_score` 降順、同 score なら filesize 昇順で ranking する。各 task の 1 位候補を `outputs/selected_submission/` にコピーして暫定 final output を作る。

In [21]:
def solution_order_key(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("-inf")


def rank_candidates(scores_df):
    usable = scores_df[scores_df["size_ok"]].copy()
    if usable.empty:
        raise ValueError("no size-valid candidates")
    usable["score_key"] = usable["estimated_score"].fillna(-1.0)
    usable["solution_key"] = usable["solution"].map(solution_order_key)
    usable["score_rankable"] = usable["score_ready"].astype(bool)
    usable = usable.sort_values(
        ["task", "score_rankable", "score_key", "solution_key", "filesize", "solution"],
        ascending=[True, False, False, False, True, True],
    )
    usable["rank"] = usable.groupby("task").cumcount() + 1
    usable["selection_mode"] = np.where(usable["score_ready"], "scored", "fallback_unscored")
    return usable


def assert_all_tasks_have_candidates(ranked_df):
    ranked_tasks = set(ranked_df["task"].astype(int))
    missing = sorted(set(TASKS) - ranked_tasks)
    if missing:
        raise ValueError(f"no size-valid candidate for tasks: {missing[:20]} total={len(missing)}")


def copy_selection(selection_df, output_dir=FINAL_OUTPUT_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)
    for old in output_dir.glob("task*.onnx"):
        old.unlink()
    copied = []
    for _, row in selection_df.sort_values("task").iterrows():
        task_num = int(row["task"])
        src = ROOT / row["onnx"]
        dst = output_dir / f"task{task_num:03d}.onnx"
        shutil.copy2(src, dst)
        copied.append({
            "task": task_num,
            "solution": row["solution"],
            "rank": int(row["rank"]),
            "selection_mode": row["selection_mode"],
            "estimated_score": row["estimated_score"],
            "src": rel(src),
            "dst": rel(dst),
        })
    copied_df = pd.DataFrame(copied)
    if len(copied_df) != TASK_COUNT:
        raise ValueError(f"expected {TASK_COUNT} files, copied {len(copied_df)}")
    return copied_df


ranked = rank_candidates(scores)
assert_all_tasks_have_candidates(ranked)
unscored_tasks = sorted(
    ranked.loc[
        ranked["rank"].eq(1) & ranked["selection_mode"].eq("fallback_unscored"),
        "task",
    ].astype(int).tolist()
)
if unscored_tasks:
    print(f"fallback without score for tasks: {unscored_tasks}")
initial_selection = ranked[ranked["rank"] == 1].copy()
selection_log = copy_selection(initial_selection)
selection_log.to_csv(SELECTION_CSV, index=False)
print(f"initial output files: {len(selection_log)}")
selection_log.head()


fallback without score for tasks: [74, 115, 117, 146, 286, 287, 325, 398]
initial output files: 400


,task,solution,rank,selection_mode,estimated_score,src,dst
0,1,6332.87,1,scored,17.047385,solution/6332.87/task001.onnx,outputs/selected_submission/task001.onnx
1,2,6332.87,1,scored,13.514518,solution/6332.87/task002.onnx,outputs/selected_submission/task002.onnx
2,3,6332.87,1,scored,17.918291,solution/6332.87/task003.onnx,outputs/selected_submission/task003.onnx
3,4,6332.87,1,scored,14.084548,solution/6332.87/task004.onnx,outputs/selected_submission/task004.onnx
4,5,6332.87,1,scored,13.509903,solution/6332.87/task005.onnx,outputs/selected_submission/task005.onnx


## Verify Mode

`VERIFY_MODE = "skip"` なら full test を行わず、score ranking で選んだ 400 個をそのまま使う。`VERIFY_MODE = "full"` にすると最終 output を検証して、失敗 task を next candidate に置き換える。


In [22]:
def selected_as_repair_log(selection_df):
    rows = []
    for _, row in selection_df.sort_values("task").iterrows():
        rows.append({
            "task": int(row["task"]),
            "solution": row["solution"],
            "rank": int(row["rank"]),
            "selection_mode": row.get("selection_mode", "scored"),
            "estimated_score": row["estimated_score"],
            "candidate_onnx": row["src"],
            "onnx": row["dst"],
            "test_ready": None,
            "test_error": "skipped by VERIFY_MODE=skip",
        })
    repair_log = pd.DataFrame(rows)
    repair_log.to_csv(REPAIR_LOG_CSV, index=False)
    return repair_log


def verify_subset(session, examples):
    passed = 0
    failed = 0
    skipped = 0
    first_failure = None
    for idx, example in enumerate(examples):
        benchmark = ng.convert_to_numpy(example)
        if benchmark is None:
            skipped += 1
            continue
        try:
            actual = ng.run_network(session, benchmark["input"])
            if np.array_equal(actual, benchmark["output"]):
                passed += 1
            else:
                failed += 1
                if first_failure is None:
                    first_failure = {
                        "idx": idx,
                        "expected": example["output"],
                        "actual": ng.convert_from_numpy(actual),
                    }
        except Exception as exc:
            failed += 1
            if first_failure is None:
                first_failure = {"idx": idx, "error": repr(exc), "traceback": traceback.format_exc()}
    return passed, failed, skipped, first_failure


def verify_onnx(path: Path, source: str):
    started = time.time()
    task_num = task_num_from_path(path)
    result = {
        "task": task_num,
        "source": source,
        "onnx": rel(path),
        "test_ready": False,
        "arc_agi_pass": 0,
        "arc_agi_fail": 0,
        "arc_agi_skip": 0,
        "arc_gen_pass": 0,
        "arc_gen_fail": 0,
        "arc_gen_skip": 0,
        "test_error": None,
        "first_failure": None,
        "test_seconds": None,
    }
    try:
        model = onnx.load(path)
        onnx.checker.check_model(model, full_check=True)
        sanitized = ng.sanitize_model(copy.deepcopy(model))
        if sanitized is None:
            raise ValueError("sanitize_model failed")
        session = make_profile_session(sanitized, PROFILE_DIR / f"verify_{source}_task{task_num:03d}")
        examples = load_examples(task_num)
        agi_pass, agi_fail, agi_skip, agi_failure = verify_subset(session, examples["train"] + examples["test"])
        gen_pass, gen_fail, gen_skip, gen_failure = verify_subset(session, examples["arc-gen"])
        _ = session.end_profiling()
        result.update({
            "arc_agi_pass": agi_pass,
            "arc_agi_fail": agi_fail,
            "arc_agi_skip": agi_skip,
            "arc_gen_pass": gen_pass,
            "arc_gen_fail": gen_fail,
            "arc_gen_skip": gen_skip,
            "test_ready": agi_fail == 0 and gen_fail == 0,
            "first_failure": agi_failure or gen_failure,
        })
    except Exception as exc:
        result["test_error"] = repr(exc)
        result["first_failure"] = traceback.format_exc()
    finally:
        result["test_seconds"] = time.time() - started
    return result


def candidate_for_rank(ranked_df, task_num, rank):
    rows = ranked_df[(ranked_df["task"] == task_num) & (ranked_df["rank"] == rank)]
    if rows.empty:
        return None
    return rows.iloc[0]


def repair_final_output(ranked_df, output_dir=FINAL_OUTPUT_DIR, max_rank=None):
    current_rank = {task: 1 for task in TASKS}
    repair_rows = []
    max_available_rank = int(ranked_df["rank"].max()) if max_rank is None else int(max_rank)

    for task_num in TASKS:
        while True:
            rank = current_rank[task_num]
            candidate = candidate_for_rank(ranked_df, task_num, rank)
            if candidate is None:
                raise ValueError(f"task {task_num:03d}: no candidate up to rank {rank}")

            src = ROOT / candidate["onnx"]
            dst = output_dir / f"task{task_num:03d}.onnx"
            shutil.copy2(src, dst)

            print(f"task {task_num:03d}: testing {candidate['solution']} rank={rank}")
            checked = verify_onnx(dst, source=f"{candidate['solution']}_rank{rank}")
            repair_rows.append({
                "task": task_num,
                "solution": candidate["solution"],
                "rank": rank,
                "selection_mode": candidate.get("selection_mode", "scored"),
                "estimated_score": candidate["estimated_score"],
                "candidate_onnx": candidate["onnx"],
                **checked,
            })

            if checked["test_ready"]:
                print(
                    f"  ok agi={checked['arc_agi_pass']} gen={checked['arc_gen_pass']} "
                    f"time={checked['test_seconds']:.1f}s"
                )
                break

            print(f"  failed; trying next candidate. error={checked['test_error']}")
            current_rank[task_num] += 1
            task_max_rank = int(ranked_df.loc[ranked_df["task"] == task_num, "rank"].max())
            if current_rank[task_num] > min(max_available_rank, task_max_rank):
                print(f"  exhausted candidates for task {task_num:03d}; keeping last tried candidate")
                break

    repair_log = pd.DataFrame(repair_rows)
    repair_log.to_csv(REPAIR_LOG_CSV, index=False)
    return repair_log


if VERIFY_MODE == "skip":
    print("VERIFY_MODE=skip: full test is skipped; using score-ranked selected output as-is.")
    repair_log = selected_as_repair_log(selection_log)
elif VERIFY_MODE == "full":
    repair_log = repair_final_output(ranked)
else:
    raise ValueError(f"unknown VERIFY_MODE: {VERIFY_MODE}")
repair_log.tail()


VERIFY_MODE=skip: full test is skipped; using score-ranked selected output as-is.


,task,solution,rank,selection_mode,estimated_score,candidate_onnx,onnx,test_ready,test_error
395,396,6332.87,1,scored,14.009399,solution/6332.87/task396.onnx,outputs/selected_submission/task396.onnx,None,skipped by VERIFY_MODE=skip
396,397,6332.87,1,scored,15.045772,solution/6332.87/task397.onnx,outputs/selected_submission/task397.onnx,None,skipped by VERIFY_MODE=skip
397,398,6332.87,1,fallback_unscored,NaN,solution/6332.87/task398.onnx,outputs/selected_submission/task398.onnx,None,skipped by VERIFY_MODE=skip
398,399,6332.87,1,scored,17.923346,solution/6332.87/task399.onnx,outputs/selected_submission/task399.onnx,None,skipped by VERIFY_MODE=skip
399,400,6332.87,1,scored,14.532535,solution/6332.87/task400.onnx,outputs/selected_submission/task400.onnx,None,skipped by VERIFY_MODE=skip


## Create submission.zip

`outputs/selected_submission/` から `outputs/submission.zip` を作成する。`VERIFY_MODE="skip"` の場合は score 最優先で集めた output をそのまま zip 化する。


In [23]:
def create_submission_zip(source_dir=FINAL_OUTPUT_DIR, zip_path=FINAL_ZIP):
    files = sorted(source_dir.glob("task*.onnx"))
    if len(files) != TASK_COUNT:
        raise ValueError(f"expected {TASK_COUNT} ONNX files, found {len(files)}")
    expected_names = {f"task{i:03d}.onnx" for i in TASKS}
    actual_names = {p.name for p in files}
    missing = sorted(expected_names - actual_names)
    extra = sorted(actual_names - expected_names)
    if missing or extra:
        raise ValueError(f"invalid submission files. missing={missing[:10]} extra={extra[:10]}")

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in files:
            zf.write(path, arcname=path.name)
    return {
        "zip": rel(zip_path),
        "files": len(files),
        "bytes": zip_path.stat().st_size,
    }


zip_info = create_submission_zip()
zip_info

{'zip': 'outputs/submission.zip', 'files': 400, 'bytes': 705187}

## Summary

In [24]:
final_selection = repair_log.sort_values(["task", "rank"]).groupby("task").tail(1).copy()
if "test_ready" in final_selection:
    failed = final_selection[final_selection["test_ready"].eq(False)]
else:
    failed = pd.DataFrame()
replaced = final_selection[final_selection["rank"] > 1]

print(f"verify mode: {VERIFY_MODE}")
print(f"solutions: {len(solution_summary)}")
print(f"scored candidates: {len(scores)}")
print(f"final files: {len(final_selection)}")
print(f"replaced by next candidate: {len(replaced)}")
print(f"failures: {len(failed)}")
print(f"submission: {zip_info['zip']} ({zip_info['bytes']:,} bytes)")
print(f"score cache: {rel(SCORE_CACHE_CSV)}")
print(f"selection csv: {rel(SELECTION_CSV)}")
print(f"repair log: {rel(REPAIR_LOG_CSV)}")

display_cols = [
    "task", "solution", "rank", "selection_mode", "estimated_score",
    "test_ready", "candidate_onnx", "onnx",
]
display_cols = [col for col in display_cols if col in final_selection.columns]
final_selection[display_cols].head(20)


verify mode: skip
solutions: 4
scored candidates: 1600
final files: 400
replaced by next candidate: 0
failures: 0
submission: outputs/submission.zip (705,187 bytes)
score cache: artifacts/submission-builder/candidate_scores.csv
selection csv: artifacts/submission-builder/selected_candidates.csv
repair log: artifacts/submission-builder/repair_log.csv


,task,solution,rank,selection_mode,estimated_score,test_ready,candidate_onnx,onnx
0,1,6332.87,1,scored,17.047385,None,solution/6332.87/task001.onnx,outputs/selected_submission/task001.onnx
1,2,6332.87,1,scored,13.514518,None,solution/6332.87/task002.onnx,outputs/selected_submission/task002.onnx
2,3,6332.87,1,scored,17.918291,None,solution/6332.87/task003.onnx,outputs/selected_submission/task003.onnx
3,4,6332.87,1,scored,14.084548,None,solution/6332.87/task004.onnx,outputs/selected_submission/task004.onnx
4,5,6332.87,1,scored,13.509903,None,solution/6332.87/task005.onnx,outputs/selected_submission/task005.onnx
5,6,6332.87,1,scored,18.527654,None,solution/6332.87/task006.onnx,outputs/selected_submission/task006.onnx
6,7,6332.87,1,scored,16.108351,None,solution/6332.87/task007.onnx,outputs/selected_submission/task007.onnx
7,8,6332.87,1,scored,14.944307,None,solution/6332.87/task008.onnx,outputs/selected_submission/task008.onnx
8,9,6274.21,1,scored,13.694825,None,solution/6274.21/task009.onnx,outputs/selected_submission/task009.onnx
9,10,6332.87,1,scored,16.479811,None,solution/6332.87/task010.onnx,outputs/selected_submission/task010.onnx
